In [1]:
# ============================================================
# STEP 1: DATA INGESTION, GOVERNANCE & QUALITY CONTROL
# Purpose:
# Build a single, trusted source of truth for executive analytics
# ============================================================

import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings("ignore")

generation_raw = pd.read_csv("Plant_1_Generation_Data.csv")
weather_raw = pd.read_csv("Plant_1_Weather_Sensor_Data.csv")

generation = generation_raw.copy()
weather = weather_raw.copy()

generation.info()
weather.info()

generation.columns = generation.columns.str.lower()
weather.columns = weather.columns.str.lower()

generation['date_time'] = pd.to_datetime(generation['date_time'])
weather['date_time'] = pd.to_datetime(weather['date_time'])

generation.isnull().sum()
weather.isnull().sum()

generation.duplicated().sum()
weather.duplicated().sum()

generation['hour'] = generation['date_time'].dt.hour

numeric_gen_cols = generation.select_dtypes(include='number').columns
generation = generation[(generation[numeric_gen_cols] >= 0).all(axis=1)]

numeric_weather_cols = weather.select_dtypes(include='number').columns
weather = weather[(weather[numeric_weather_cols] >= 0).all(axis=1)]

suspicious_zero_output = generation[
    (generation['hour'].between(6, 18)) &
    (generation['ac_power'] == 0)
]

suspicious_zero_output.shape

solar_df = pd.merge(
    generation,
    weather,
    on=['date_time','plant_id'],
    how='inner'
)

solar_df.shape
solar_df.head()
solar_df.info()

solar_df.rename(columns={
    "source_key_x": "inverter_id",
    "source_key_y": "weather_sensor_id"
}, inplace=True)

# ============================================================
# 🔧 FIX 1: REMOVE NON-OPERATIONAL ROWS
# (THIS fixes 90% loss & blank KPIs)
# ============================================================
solar_df = solar_df[
    (solar_df['dc_power'] > 0) &
    (solar_df['ac_power'] > 0)
]

# ============================================================
# 🔧 FIX 2: CLARIFY IRRADIATION UNIT (NO VALUE CHANGE)
# Irradiation is already in kW/m² (0–1.2 range)
# ============================================================
solar_df.rename(columns={
    "irradiation": "solar_irradiation_kwm2"
}, inplace=True)

# ============================================================
# EXISTING CALCULATIONS (KEEP – diagnostic only)
# ============================================================
solar_df['efficiency_pct'] = np.where(
    solar_df['dc_power'] > 0,
    (solar_df['ac_power'] / solar_df['dc_power']) * 100,
    np.nan
)

solar_df['power_loss_kw'] = solar_df['dc_power'] - solar_df['ac_power']

solar_df['temp_diff'] = (
    solar_df['module_temperature'] - solar_df['ambient_temperature']
)

solar_df['specific_yield'] = np.where(
    solar_df['solar_irradiation_kwm2'] > 0,
    solar_df['dc_power'] / solar_df['solar_irradiation_kwm2'],
    np.nan
)

# ============================================================
# 🔧 FIX 3: PHYSICS-BASED SANITY FILTERS
# (Stops unrealistic efficiency & loss)
# ============================================================
solar_df = solar_df[
    (solar_df['efficiency_pct'].between(0, 30))
]

# ============================================================
# SAVE CLEAN DATA FOR DASHBOARD
# ============================================================
solar_df.to_csv(
    "solar_plant_cleaned_data.csv",
    index=False
)

solar_df.shape


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68778 entries, 0 to 68777
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATE_TIME    68778 non-null  object 
 1   PLANT_ID     68778 non-null  int64  
 2   SOURCE_KEY   68778 non-null  object 
 3   DC_POWER     68778 non-null  float64
 4   AC_POWER     68778 non-null  float64
 5   DAILY_YIELD  68778 non-null  float64
 6   TOTAL_YIELD  68778 non-null  float64
dtypes: float64(4), int64(1), object(2)
memory usage: 3.7+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3182 entries, 0 to 3181
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   DATE_TIME            3182 non-null   object 
 1   PLANT_ID             3182 non-null   int64  
 2   SOURCE_KEY           3182 non-null   object 
 3   AMBIENT_TEMPERATURE  3182 non-null   float64
 4   MODULE_TEMPERATURE   3182 non-null   floa

(36823, 16)